In [1]:
!source ~/cBottle/.venv/bin/activate
!pip install ipykernel --break-system-packages
!python3 -m ipykernel install --user --name cbottle-venv --display-name "cBottle (.venv)"


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Installed kernelspec cbottle-venv in /home/k/k202223/.local/share/jupyter/kernels/cbottle-venv


In [2]:
import xarray as xr
!module load python3/2025.01-gcc-13.3.0

In [3]:
root = "/work/bk1444/climbench/data"

In [4]:
ds = xr.open_zarr("/work/bk1444/climbench/data/experiment3/healpix/input_train.zarr")
print(dict(ds.sizes), list(ds.data_vars))

{'time': 21915, 'cell': 768} ['msl', 'q850', 'tas', 'tcc', 'tp', 'u850', 'v850', 'z500']


In [5]:
ds

<xarray.Dataset> Size: 539MB
Dimensions:    (time: 21915, cell: 768)
Coordinates:
  * time       (time) datetime64[ns] 175kB 1940-01-01 1940-01-02 ... 1999-12-31
  * cell       (cell) int64 6kB 0 1 2 3 4 5 6 7 ... 761 762 763 764 765 766 767
    latitude   (cell) float64 6kB ...
    longitude  (cell) float64 6kB ...
    crs        float64 8B ...
Data variables:
    msl        (time, cell) float32 67MB ...
    q850       (time, cell) float32 67MB ...
    tas        (time, cell) float32 67MB ...
    tcc        (time, cell) float32 67MB ...
    tp         (time, cell) float32 67MB ...
    u850       (time, cell) float32 67MB ...
    v850       (time, cell) float32 67MB ...
    z500       (time, cell) float32 67MB ...
Attributes: (12/13)
    CDI:                               Climate Data Interface version 2.2.4 (...
    CDO:                               Climate Data Operators version 2.2.2 (...
    Conventions:                       CF-1.6
    derived_from:                      target
    grid_doctor_coarsened_from_level:  7
    grid_doctor_method:                conservative
    ...                                ...
    healpix_level:                     3
    healpix_nside:                     8
    healpix_order:                     nested
    history:                           Tue Jun 23 08:44:29 2026: cdo -O -f nc...
    institution:                       European Centre for Medium-Range Weath...
    mapping_type:                      coarse_field_to_high_resolution_field

In [6]:
!pwd

/home/k/k202223/cBottle


In [7]:
!cd /home/k/k202223/cBottle

In [8]:
!ls

CHANGELOG.md	     Dockerfile   output			     scripts
checkpoints	     docs	  pyproject.toml		     site
ci		     examples	  README.md			     src
conftest.py	     LICENSE.txt  run_inference_coarse.sh	     tests
CONTRIBUTING.md      logs	  run_inference_super_resolution.sh  wandb
data_analysis.ipynb  Makefile	  run.sh
docker		     mkdocs.yml   run_train_coarse.sh


In [9]:
import zarr
import numpy as np

def compute_mean_std_streaming(path, field, batch_size=500):
    group = zarr.open_group(path, mode="r")
    arr = group[field]
    n_time = arr.shape[0]
    total_sum = 0.0
    total_sumsq = 0.0
    total_count = 0
    for start in range(0, n_time, batch_size):
        end = min(start + batch_size, n_time)
        chunk = arr[start:end].astype(np.float64)  # only this slice hits memory
        total_sum += chunk.sum()
        total_sumsq += (chunk ** 2).sum()
        total_count += chunk.size
    mean = total_sum / total_count
    var = max(total_sumsq / total_count - mean**2, 0.0)  # guard against float cancellation
    return mean, np.sqrt(var)

path = "/work/bk1444/climbench/data/experiment3/healpix/target_train.zarr"
fields = ["msl", "q850", "tas", "tcc", "tp", "u850", "v850", "z500"]
for field in fields:
    m, s = compute_mean_std_streaming(path, field)
    print(f'    "{field}": mean={m}, scale={s}')

    "msl": mean=101139.39094338276, scale=1062.9293798393428
    "q850": mean=0.005964962388804392, scale=0.004028018112375323
    "tas": mean=286.99915542783003, scale=15.414200571501576
    "tcc": mean=0.6229747817337858, scale=0.3009191133842002
    "tp": mean=0.0028682559418698973, scale=0.006175653414683115
    "u850": mean=1.1146329207266692, scale=7.60048045817027
    "v850": mean=0.09364891663880362, scale=5.052339021547254
    "z500": mean=55433.52487005346, scale=2704.6219155593317
